In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
DATA_PATH = "../data/raw/LC_loans_granting_model_dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

In [ ]:
print("DATASET DIMENSIONS")
print(df.shape)

In [ ]:
print("\nVARIABLES AND TYPES")
print(df.dtypes.to_string())

In [ ]:
print("\nMISSING VALUES")
print(df.isna().sum().to_string())

In [ ]:
print("\nTARGET DISTRIBUTION")
print(df["Default"].value_counts(dropna=False))
print(df["Default"].value_counts(normalize=True, dropna=False))

In [ ]:
print("\nDATA COVERAGE")
dates = pd.to_datetime(df["issue_d"], format="%b-%Y")
print("Earliest:", dates.min())
print("Latest:", dates.max())

In [ ]:
print("\nLOANS BY YEAR")
print(dates.dt.year.value_counts().sort_index())

In [ ]:
print("\nDEFAULT RATE BY YEAR")
annual_default = (
    df.assign(year=dates.dt.year)
    .groupby("year")["Default"]
    .agg(["count","mean"])
)

annual_default["mean"] *= 100
annual_default = annual_default.rename(columns={"count":"loans", "mean":"default_rate_pct"})

print(annual_default.round(2).to_string())

In [ ]:
print("\nFEATURE MEANS BY YEAR")

annual_features = (
    df.assign(year=dates.dt.year)
    .groupby("year")[["fico_n","dti_n","loan_amnt"]]
    .mean()
)

print(annual_features.round(2).to_string())

In [ ]:
fico_bins = [300, 600, 650, 700, 750, 800, 850]

fico_data = df.copy()
fico_data["fico_group"] = pd.cut(
    fico_data["fico_n"],
    bins=fico_bins,
    include_lowest=True
)

fico_default = (
    fico_data.groupby("fico_group", observed=True)["Default"]
    .agg(["mean", "count"])
)

fico_default["mean"] *= 100

display(fico_default)

fico_default["mean"].plot(
    kind="bar",
    figsize=(9, 4),
    ylabel="Default rate (%)",
    xlabel="FICO range",
    title="Observed default rate by FICO range",
    rot=45
)

plt.tight_layout()
plt.show()

In [ ]:
# Define origination periods
periods = pd.cut(
    dates.dt.year,
    bins=[2006, 2012, 2015, 2017, 2018],
    labels=["2007–2012", "2013–2015", "2016–2017", "2018"]
)

# Calculate default rates by FICO range and period
comparison = (
    df.assign(period=periods, fico_group=fico_data["fico_group"])
    .groupby(["fico_group", "period"], observed=True)["Default"]
    .agg(["mean", "count"])
)

comparison["mean"] *= 100

display(comparison)

In [ ]:
ax = (
    comparison["mean"]
    .unstack("period")
    .plot(
        kind="bar",
        figsize=(11, 5),
        ylabel="Default rate (%)",
        xlabel="FICO range",
        title="Default rates by FICO range and origination period",
        rot=45,
    )
)

ax.legend(title="Origination period")
plt.tight_layout()
plt.show()